**Lab:** Lab 6  
# Deep Learning for NLP

**Name:** Mohammad Buhamad

**Student ID:** [2240005142]

In [7]:
import pandas as pd

df = pd.read_csv(
    "07-yelp-dataset.txt",
    sep="\t",
    names=["review", "label"]
)

print("Number of reviews:", len(df))

df.head()

Number of reviews: 1000


,review,label
0,Wow... Loved this place.,1
1,Crust is not good.,0
2,Not tasty and the texture was just nasty.,0
3,Stopped by during the late May bank holiday of...,1
4,The selection on the menu was great and so wer...,1


In [8]:
print(df["label"].value_counts())

label
1    500
0    500
Name: count, dtype: int64


In [9]:
import re

def preprocess_text(text):

    # 1. Convert to lowercase
    text = text.lower()

    # 2. Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # 3. Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df["review"] = df["review"].apply(preprocess_text)

df.head()

,review,label
0,wow loved this place,1
1,crust is not good,0
2,not tasty and the texture was just nasty,0
3,stopped by during the late may bank holiday of...,1
4,the selection on the menu was great and so wer...,1


In [10]:
from sklearn.model_selection import train_test_split

X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 800
Testing reviews: 200


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=1000, lowercase=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)

Training TF-IDF shape: (800, 1000)
Testing TF-IDF shape: (200, 1000)


In [12]:
!pip install torch
import torch
import torch.nn as nn

X_train_tensor = torch.FloatTensor(
    X_train_tfidf.toarray()
)

X_test_tensor = torch.FloatTensor(
    X_test_tfidf.toarray()
)

y_train_tensor = torch.FloatTensor(
    y_train.values
).reshape(-1, 1)

   ---------------------------------------- 0.0/124.1 MB ? eta -:--:--
   ---------------------------------------- 0.1/124.1 MB 1.6 MB/s eta 0:01:18
   ---------------------------------------- 0.1/124.1 MB 1.6 MB/s eta 0:01:18
   ---------------------------------------- 0.1/124.1 MB 558.5 kB/s eta 0:03:43
   ---------------------------------------- 0.1/124.1 MB 708.1 kB/s eta 0:02:56
   ---------------------------------------- 0.2/124.1 MB 903.5 kB/s eta 0:02:18
   ---------------------------------------- 0.2/124.1 MB 769.9 kB/s eta 0:02:41
   ---------------------------------------- 0.3/124.1 MB 916.0 kB/s eta 0:02:16
   ---------------------------------------- 0.4/124.1 MB 1.1 MB/s eta 0:01:55
   ---------------------------------------- 0.5/124.1 MB 1.2 MB/s eta 0:01:45
   ---------------------------------------- 0.6/124.1 MB 1.2 MB/s eta 0:01:42
   ---------------------------------------- 0.8/124.1 MB 1.6 MB/s eta 0:01:20
   ---------------------------------------- 0.9/124.1 MB 1.7 

In [13]:
input_size = X_train_tensor.shape[1]

model = nn.Sequential(
    nn.Linear(input_size, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 1)
)

print(model)

Sequential(
  (0): Linear(in_features=1000, out_features=64, bias=True)
  (1): ReLU()
  (2): Linear(in_features=64, out_features=32, bias=True)
  (3): ReLU()
  (4): Linear(in_features=32, out_features=1, bias=True)
)


In [14]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.01
)

In [15]:
epochs = 10

for epoch in range(epochs):

    model.train()

    # 1. Forward pass
    outputs = model(X_train_tensor)

    # 2. Calculate loss
    loss = loss_function(outputs, y_train_tensor)

    # 3. Backpropagation
    loss.backward()

    # 4. Update model weights
    optimizer.step()

    if (epoch + 1) % 5 == 0:
        print(
            f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}"
        )

Epoch 5/10 - Loss: 0.6582
Epoch 10/10 - Loss: 0.4963


In [16]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():

    test = model(X_test_tensor)

    test_probabilities = torch.sigmoid(test)

    predictions = (
        test_probabilities >= 0.5
    ).int().numpy().flatten()

accuracy = accuracy_score(
    y_test,
    predictions
)

print(f"Test Accuracy: {accuracy:.3f}")

Test Accuracy: 0.725


In [17]:
print("Classification Report:")

print(
    classification_report(
        y_test,
        predictions,
        target_names=["Negative", "Positive"]
    )
)

print("Confusion Matrix:")

print(
    confusion_matrix(
        y_test,
        predictions
    )
)

Classification Report:
              precision    recall  f1-score   support

    Negative       0.65      0.95      0.77        96
    Positive       0.92      0.52      0.66       104

    accuracy                           0.72       200
   macro avg       0.78      0.73      0.72       200
weighted avg       0.79      0.72      0.71       200

Confusion Matrix:
[[91  5]
 [50 54]]


In [18]:
!pip install gensim
import gensim
from nltk.tokenize import word_tokenize

In [19]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\xmoha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\xmoha\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [20]:
train_tokens = [
    word_tokenize(review)
    for review in X_train
]

test_tokens = [
    word_tokenize(review)
    for review in X_test
]

In [21]:
import numpy as np

def review_to_vector(tokens, model):

    vectors = []

    for word in tokens:

        if word in model.wv:
            vectors.append(model.wv[word])

    if len(vectors) == 0:
        return np.zeros(model.vector_size)

    return np.mean(vectors, axis=0)

In [23]:
import gensim

w2v_model = gensim.models.Word2Vec(
    train_tokens,
    vector_size=200,
    window=5,
    min_count=1,
    sg=1
)

In [24]:
X_train_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in train_tokens
])

X_test_w2v = np.array([
    review_to_vector(tokens, w2v_model)
    for tokens in test_tokens
])

In [25]:
X_train_w2v_tensor = torch.FloatTensor(X_train_w2v)

X_test_w2v_tensor = torch.FloatTensor(X_test_w2v)

y_train_w2v_tensor = torch.FloatTensor(
    y_train.values
).reshape(-1, 1)

In [26]:
print("Training Word2Vec shape:", X_train_w2v_tensor.shape)
print("Testing Word2Vec shape:", X_test_w2v_tensor.shape)

Training Word2Vec shape: torch.Size([800, 200])
Testing Word2Vec shape: torch.Size([200, 200])


In [27]:
input_size = X_train_w2v_tensor.shape[1]

model_w2v = nn.Sequential(

    nn.Linear(input_size, 128),
    nn.ReLU(),

    nn.Linear(128, 64),
    nn.ReLU(),

    nn.Linear(64, 32),
    nn.ReLU(),

    nn.Linear(32, 1)
)

print(model_w2v)

Sequential(
  (0): Linear(in_features=200, out_features=128, bias=True)
  (1): ReLU()
  (2): Linear(in_features=128, out_features=64, bias=True)
  (3): ReLU()
  (4): Linear(in_features=64, out_features=32, bias=True)
  (5): ReLU()
  (6): Linear(in_features=32, out_features=1, bias=True)
)


In [28]:
loss_function = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(
    model_w2v.parameters(),
    lr=0.001
)

In [29]:
epochs = 15

for epoch in range(epochs):

    model_w2v.train()

    # Forward pass
    outputs = model_w2v(X_train_w2v_tensor)

    # Calculate loss
    loss = loss_function(
        outputs,
        y_train_w2v_tensor
    )

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Clear gradients
    optimizer.zero_grad()

    print(
        f"Epoch {epoch + 1}/{epochs} - Loss: {loss.item():.4f}"
    )

Epoch 1/15 - Loss: 0.6932
Epoch 2/15 - Loss: 0.6932
Epoch 3/15 - Loss: 0.6931
Epoch 4/15 - Loss: 0.6931
Epoch 5/15 - Loss: 0.6931
Epoch 6/15 - Loss: 0.6931
Epoch 7/15 - Loss: 0.6931
Epoch 8/15 - Loss: 0.6931
Epoch 9/15 - Loss: 0.6931
Epoch 10/15 - Loss: 0.6930
Epoch 11/15 - Loss: 0.6930
Epoch 12/15 - Loss: 0.6930
Epoch 13/15 - Loss: 0.6930
Epoch 14/15 - Loss: 0.6930
Epoch 15/15 - Loss: 0.6930


In [30]:
model_w2v.eval()

with torch.no_grad():

    test_outputs = model_w2v(
        X_test_w2v_tensor
    )

    test_probabilities = torch.sigmoid(
        test_outputs
    )

    predictions_w2v = (
        test_probabilities >= 0.5
    ).int().numpy().flatten()

accuracy_w2v = accuracy_score(
    y_test,
    predictions_w2v
)

print(f"Test Accuracy: {accuracy_w2v:.3f}")

Test Accuracy: 0.480
